# Satellite Images Annotation


---
## 0 · Imports & Configuration


In [ ]:
import os, warnings
import numpy as np
import rasterio
from pathlib import Path
from scipy.ndimage import uniform_filter
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import ListedColormap, BoundaryNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
import pandas as pd
warnings.filterwarnings('ignore')

# paths
AREA_NAME = "Wuchang District, China"
DATE_STR  = "2020-07-23"
BASE      = r"C:\Users\sultan\Desktop\data generation\flood_data_exports\Wuchang_District_China\2020-07-23"
SAR_DIR   = Path(BASE) / "SAR_ALL_BANDS"
OPT_DIR   = Path(BASE) / "OPTICAL_ALL_BANDS"
MASK_DIR  = Path(BASE) / "QC_WEAK_SUPERVISION"
MASK_DIR.mkdir(parents=True, exist_ok=True)

# hyperparameters (must mirror main.py in Data Acquisition)
S1_DB_MIN, S1_DB_MAX   = -30.0,  5.0
S1_WATER_VH_THRESH     = -20.0
LEE_ENL, LEE_WINDOW    =   4.4,  7
KI_MAX_ITER            =  50
SDWI_THRESH            =  17.5
HOT_THRESH             =   0.05
BLUE_SWIR_THRESH       =   2.5
NIR_DROP_THRESH        =   0.10
MNDWI_THRESH           =   0.2   # standard S2 MNDWI (B3-B11)/(B3+B11) threshold
AWEI_NSH_THRESH        =   0.0
AWEI_SH_THRESH         =   0.0
NDVI_VEG_THRESH        =   0.3

BAND_NAMES = ["B1","B2","B3","B4","B5","B6","B7",
              "B8","B8A","B9","B10","B11","B12"]
BG = "#0d0d1a"
print("Config loaded")
print(f"SAR : {SAR_DIR}")
print(f"OPT : {OPT_DIR}")

---
## 1 · Tile Inventory


In [ ]:
sar_tiles = sorted(SAR_DIR.glob("*.tif"))
opt_tiles = sorted(OPT_DIR.glob("*.tif"))
sar_ids   = {int(p.stem.split("_")[1]) for p in sar_tiles}
opt_ids   = {int(p.stem.split("_")[1]) for p in opt_tiles}
paired    = sorted(sar_ids & opt_ids)
sar_only  = sorted(sar_ids - opt_ids)
opt_only  = sorted(opt_ids - sar_ids)


print(f"SAR  tiles         : {len(sar_tiles):>5}")
print(f"Optical tiles      : {len(opt_tiles):>5}")
print(f"Paired (SAR+OPT)  : {len(paired):>5}")
print(f"SAR-only          : {len(sar_only):>5}")
print(f"OPT-only          : {len(opt_only):>5}")

if sar_tiles:
    with rasterio.open(sar_tiles[0]) as s:
        print(f"\nSAR  tile[0]  bands={s.count}  shape={s.shape}  dtype={s.dtypes[0]}  CRS=EPSG:{s.crs.to_epsg()}")
if opt_tiles:
    with rasterio.open(opt_tiles[0]) as s:
        b1 = s.read(1).astype(np.float32)
        valid = b1[b1 > 0]
        print(f"OPT  tile[0]  bands={s.count}  shape={s.shape}  dtype={s.dtypes[0]}  "
              f"val_pct={100*valid.size/max(b1.size,1):.1f}%  range=[{b1.min():.0f},{b1.max():.0f}]")

---
## 2 · Core Helper Functions


In [3]:
# stretch and colormap helpers

def robust_stretch(arr, nodata_mask=None, lo=2, hi=98):
    """
    Percentile stretch with rasterio nodata handling.
    Nodata and zero-fill pixels become NaN so swath edges render as
    clean black instead of being pulled into the valid colour range.
    Applies gamma=0.5 for natural brightness on valid pixels.
    """
    a = arr.astype(np.float32).copy()
    if nodata_mask is not None:       # rasterio dataset_mask: 255=valid, 0=nodata
        a[nodata_mask == 0] = np.nan
    a[a == 0] = np.nan                # zero-fill fallback
    a[~np.isfinite(a)] = np.nan
    valid = a[np.isfinite(a)]
    if valid.size < 10:
        return np.full_like(a, np.nan)
    mn, mx = np.percentile(valid, [lo, hi])
    if mx - mn < 1e-6:
        return np.full_like(a, np.nan)
    out = np.clip((a - mn) / (mx - mn), 0, 1)
    out = np.power(out, 0.5)          # gamma=0.5
    return out.astype(np.float32)

def db_stretch(arr):
    """Linear dB scale [-30, +5] to [0, 1]"""
    return np.clip((arr - S1_DB_MIN) / (S1_DB_MAX - S1_DB_MIN), 0, 1).astype(np.float32)

def make_rgb_opt(bands, nodata_mask=None):
    """True-color B4/B3/B2, with nodata pixels filled black."""
    r = robust_stretch(bands.get("B4", np.zeros((2,2))), nodata_mask)
    g = robust_stretch(bands.get("B3", np.zeros((2,2))), nodata_mask)
    b = robust_stretch(bands.get("B2", np.zeros((2,2))), nodata_mask)
    rgb = np.stack([r, g, b], axis=-1).astype(np.float32)
    return np.nan_to_num(rgb, nan=0.0)

def make_fc_opt(bands, nodata_mask=None):
    """
    False-color composite, B12/B8/B4 (SWIR=Red, NIR=Green, Red=Blue).
    Water appears dark blue, vegetation bright green, urban/bare magenta-pink.
    """
    r = robust_stretch(bands.get("B12", bands.get("B11", np.zeros((2,2)))), nodata_mask)
    g = robust_stretch(bands.get("B8",  bands.get("B4",  np.zeros((2,2)))), nodata_mask)
    b = robust_stretch(bands.get("B4",  np.zeros((2,2))), nodata_mask)
    rgb = np.stack([r, g, b], axis=-1).astype(np.float32)
    return np.nan_to_num(rgb, nan=0.0)

def make_rgb_sar(vv_f, vh_f):
    """VV=Red VH=Green Blue=0 composite (dB stretched)"""
    return np.stack([db_stretch(vv_f), db_stretch(vh_f),
                     np.zeros_like(vv_f)], axis=-1).astype(np.float32)

def add_colorbar(fig, ax, im, label, ticklabels=None, ticks=None):
    div = make_axes_locatable(ax)
    cax = div.append_axes("right", size="5%", pad=0.04)
    cb  = fig.colorbar(im, cax=cax)
    cb.set_label(label, color="white", fontsize=6)
    if ticks is not None:
        cb.set_ticks(ticks)
    if ticklabels is not None:
        cb.set_ticklabels(ticklabels)
    cb.ax.yaxis.set_tick_params(labelsize=5.5, labelcolor="white", color="white")
    cb.outline.set_edgecolor("#555")
    return cb

def load_opt_bands(tif_path):
    """Load all S2 bands plus the rasterio dataset mask (255=valid, 0=nodata)."""
    with rasterio.open(tif_path) as src:
        bands = {n: src.read(i).astype(np.float32)
                 for i, n in enumerate(BAND_NAMES, 1) if i <= src.count}
        bands["__mask__"] = src.dataset_mask()
    return bands

def load_sar(tif_path):
    with rasterio.open(tif_path) as src:
        return src.read(1).astype(np.float32), src.read(2).astype(np.float32), src.profile

# discrete water-mask colormap
WATER_CMAP = ListedColormap(["#505050", "#486438", "#1e90ff"])  # nodata, land, water
WATER_NORM = BoundaryNorm([-1.5, -0.5, 0.5, 1.5], WATER_CMAP.N)

def add_water_colorbar(fig, ax, label="Label"):
    div = make_axes_locatable(ax)
    cax = div.append_axes("right", size="5%", pad=0.04)
    cb  = fig.colorbar(plt.cm.ScalarMappable(norm=WATER_NORM, cmap=WATER_CMAP), cax=cax)
    cb.set_ticks([-1, 0, 1])
    cb.set_ticklabels(["No-data", "Land", "Water"])
    cb.set_label(label, color="white", fontsize=6)
    cb.ax.yaxis.set_tick_params(labelsize=6, labelcolor="white", color="white")
    cb.outline.set_edgecolor("#555")


---
## 3 · Physics Processing Functions


In [4]:
def dynamic_lee_filter(arr):
    a64 = arr.astype(np.float64)
    lm  = uniform_filter(a64, size=LEE_WINDOW)
    lsq = uniform_filter(a64**2, size=LEE_WINDOW)
    lv  = np.maximum(lsq - lm**2, 0.0)
    cu  = 1.0 / np.sqrt(LEE_ENL)
    ci  = np.sqrt(lv) / (np.abs(lm) + 1e-10)
    W   = np.clip(1.0 - (cu/(ci+1e-10))**2, 0.0, 1.0)
    return (lm + W*(a64 - lm)).astype(np.float32)

def kittler_illingworth(arr):
    """Minimum-error thresholding (Kittler-Illingworth) on a 1D histogram."""
    valid = arr[np.isfinite(arr)].ravel()
    if valid.size < 10: return float(np.nanmean(valid))
    counts, edges = np.histogram(valid, bins=256)
    counts = counts.astype(np.float64)
    mids   = (edges[:-1]+edges[1:])/2.0
    total  = counts.sum()
    hn = counts/total; cum=np.cumsum(hn); mu=np.cumsum(hn*mids)
    sb = (mu[-1]*cum-mu)**2/(cum*(1-cum)+1e-12)
    thresh = float(mids[np.argmax(sb)])
    for _ in range(KI_MAX_ITER):
        m1,m2 = mids<=thresh, mids>thresh
        w1,w2 = counts[m1].sum(), counts[m2].sum()
        if w1<2 or w2<2: break
        mu1=(counts[m1]*mids[m1]).sum()/w1; mu2=(counts[m2]*mids[m2]).sum()/w2
        v1=max((counts[m1]*(mids[m1]-mu1)**2).sum()/w1,1e-12)
        v2=max((counts[m2]*(mids[m2]-mu2)**2).sum()/w2,1e-12)
        s1,s2=np.sqrt(v1),np.sqrt(v2); p1,p2=w1/(w1+w2),w2/(w1+w2)
        dv=v2-v1
        if abs(dv)<1e-12: break
        new_t=(0.5*(mu1+mu2)+v1*np.log(s2/s1+1e-12)/dv
               +np.log(p2/(p1+1e-12)+1e-12)*v1*v2/dv)
        if abs(new_t-thresh)<0.001: thresh=new_t; break
        thresh=new_t
    return float(np.clip(thresh,edges[0],edges[-1]))

def annotate_sar(vv_raw, vh_raw):
    vv_f = dynamic_lee_filter(vv_raw)
    vh_f = dynamic_lee_filter(vh_raw)
    ki   = kittler_illingworth(vh_f)
    # cap KI threshold between -25 and -15 dB
    eff  = float(np.clip(min(ki, S1_WATER_VH_THRESH), -25.0, -15.0))
    sdwi = (-(vv_f+vh_f)/2.0).astype(np.float32)
    # hard gate: VH below effective threshold, SDWI acts as a confidence booster
    water = (vh_f < eff) & np.isfinite(vv_f) & np.isfinite(vh_f)
    nd    = ~np.isfinite(vv_f) | ~np.isfinite(vh_f)

    # water confidence: how far below threshold + SDWI support, scaled to [0,1]
    ck = np.clip((eff - vh_f) / 10.0, 0, 1)          # 10 dB range
    cs = np.clip((sdwi - SDWI_THRESH) / 10.0, 0, 1)  # SDWI margin
    water_conf = np.where(water, (ck + cs) / 2.0, 0.0).astype(np.float32)

    # land confidence: how far above threshold VH is
    land_dist  = np.clip((vh_f - eff) / 10.0, 0, 1)

    # SAR water probability in [0,1]:
    #   water  -> 0.5 + 0.5*conf      (0.5-1.0)
    #   land   -> 0.5 - 0.5*land_dist (0.0-0.5)
    #   nodata -> 0.5 (uncertain)
    sar_prob = np.where(nd, 0.5,
               np.where(water,
                        0.5 + 0.5 * water_conf,
                        0.5 - 0.5 * land_dist)).astype(np.float32)

    lbl  = np.full(vh_f.shape, -1, dtype=np.int8)
    lbl[~nd & water]  = 1
    lbl[~nd & ~water] = 0
    meta = dict(ki_thresh=round(ki,3), eff_thresh=round(eff,3),
                sdwi_mean=round(float(np.nanmean(sdwi)),3),
                sar_water_pct=round(100*np.sum(lbl==1)/max(np.sum(lbl>=0),1),2))
    return lbl, sar_prob, meta, vv_f, vh_f, sdwi

def cloud_shadow_mask(bands):
    """
    Cloud and shadow mask tuned to avoid flagging open water.

    Cloud tests require a minimum brightness gate (b2 > 0.15) so that
    dark water pixels, which have a high B2/B11 ratio, aren't classified
    as cloud. Shadow detection excludes water-like pixels (positive simple
    NDWI), since water also has low NIR.
    """
    b2  = bands.get("B2", np.zeros((2,2)))/10000.0
    b3  = bands.get("B3", np.zeros((2,2)))/10000.0
    b4  = bands.get("B4", np.zeros((2,2)))/10000.0
    b8  = bands.get("B8", np.zeros((2,2)))/10000.0
    b11 = bands.get("B11",np.zeros((2,2)))/10000.0

    # cloud tests, gated by brightness
    bright = b2 > 0.15                              # true clouds are bright in blue
    hot       = (b2 - 0.5*b4 > HOT_THRESH) & bright
    blue_swir = (b2 / (b11+1e-8) > BLUE_SWIR_THRESH) & bright
    cloud  = hot | blue_swir

    # shadow test, excluding water-like pixels
    ndwi_simple = (b3 - b8) / (b3 + b8 + 1e-8)    # >0 = water-like
    is_water_like = ndwi_simple > 0.0
    shadow = (b8 < NIR_DROP_THRESH) & ~cloud & ~is_water_like

    clear  = ~cloud & ~shadow & np.isfinite(b2) & np.isfinite(b8)
    return cloud, shadow, clear

def annotate_optical(bands):
    b2  = bands.get("B2", np.zeros((2,2)))/10000.0
    b3  = bands.get("B3", np.zeros((2,2)))/10000.0
    b4  = bands.get("B4", np.zeros((2,2)))/10000.0
    b8  = bands.get("B8", np.zeros((2,2)))/10000.0
    b11 = bands.get("B11",np.zeros((2,2)))/10000.0
    b12 = bands.get("B12",np.zeros((2,2)))/10000.0
    cloud, shadow, clear = cloud_shadow_mask(bands)
    cfrac = float(np.mean(cloud|shadow))
    # MNDWI = (B3-B11)/(B3+B11), standard S2 formula (Xu 2006)
    # water: B3 high, B11 ~0 -> positive; land: B11>B3 -> negative
    mndwi   = np.where(b3+b11>0,(b3-b11)/(b3+b11+1e-8),0.0).astype(np.float32)
    aweinsh = (4*(b3-b11)-(0.25*b8+2.75*b12)).astype(np.float32)
    aweish  = (b2+2.5*b3-1.5*(b8+b11)-0.25*b12).astype(np.float32)
    ndvi    = np.where(b8+b4>0,(b8-b4)/(b8+b4+1e-8),0.0)
    veto    = ndvi>NDVI_VEG_THRESH
    votes   = ((mndwi>MNDWI_THRESH).astype(int)
              +(aweinsh>AWEI_NSH_THRESH).astype(int)
              +(aweish>AWEI_SH_THRESH).astype(int))
    water   = (votes>=2)&~veto&clear

    # soft probability scores for fusion, centered on each threshold so
    # score = 0.5 at the threshold, >0.5 = water evidence, <0.5 = land evidence
    sm_p = np.clip((mndwi   - MNDWI_THRESH)    / 0.30, -1, 1) * 0.5 + 0.5
    sn_p = np.clip((aweinsh - AWEI_NSH_THRESH) / 0.15, -1, 1) * 0.5 + 0.5
    ss_p = np.clip((aweish  - AWEI_SH_THRESH)  / 0.15, -1, 1) * 0.5 + 0.5
    index_score = ((sm_p + sn_p + ss_p) / 3.0).astype(np.float32)
    # 0.0 = strongly dry, 0.5 = at threshold, 1.0 = open water

    # cloud/shadow/nodata pixels are left at 0.5 (uncertain, SAR decides)
    opt_prob = np.where(~clear, 0.5, index_score).astype(np.float32)
    conf = opt_prob
    lbl=np.full(b3.shape,-1,dtype=np.int8)
    lbl[clear & water]=1; lbl[clear & ~water]=0
    meta=dict(cloud_fraction=round(cfrac,4),
              mndwi_mean=round(float(np.nanmean(mndwi[clear])),4) if clear.any() else 0,
              awei_nsh_mean=round(float(np.nanmean(aweinsh[clear])),4) if clear.any() else 0,
              opt_water_pct=round(100*np.sum(lbl==1)/max(np.sum(lbl>=0),1),2))
    idx=dict(mndwi=mndwi,aweinsh=aweinsh,aweish=aweish,ndvi=ndvi)
    return lbl,conf,cloud,shadow,clear,cfrac,meta,idx

def fuse_labels(sl, sc_prob, ol, oc_prob, cfrac):
    """
    Fuse SAR and optical water probabilities.
    sc_prob, oc_prob are water probabilities in [0,1]:
      > 0.5 = water, < 0.5 = land, = 0.5 = uncertain (nodata/cloud)
    s2_weight down-weights optical when cloudy.
    """
    s2w = (1.0 - cfrac) ** 2
    sp  = sc_prob.astype(np.float32)
    op  = oc_prob.astype(np.float32)
    # SAR always contributes, optical scaled by clarity
    wp  = (sp + op * s2w) / (1.0 + s2w)
    nd  = (sl == -1) & (ol == -1)
    fl  = np.where(nd, -1, np.where(wp > 0.5, 1, 0)).astype(np.int8)
    return fl, wp.astype(np.float32), s2w


---
## 4 · SAR Visualization — VV, VH & Composite (all tiles)
Three panels per tile: **VV band (grayscale + dB colorbar)** · **VH band (viridis + dB colorbar)** · **VV/VH composite (legend)**


In [ ]:
N = len(sar_tiles)
fig, axes = plt.subplots(N, 3, figsize=(16, 5.2*N), facecolor=BG)
if N == 1: axes = axes[np.newaxis, :]

fig.suptitle(
    f"  SAR Tiles — VV · VH · Composite\n{AREA_NAME}   ·   {DATE_STR}",
    color="white", fontsize=13, fontweight="bold", y=1.002
)

for row, tif in enumerate(sar_tiles):
    idx = int(tif.stem.split("_")[1])
    vv_r, vh_r, _ = load_sar(tif)
    vv_f = dynamic_lee_filter(vv_r)
    vh_f = dynamic_lee_filter(vh_r)

    for ax in axes[row]: ax.set_facecolor(BG)

    # Panel A: VV grayscale with dB colorbar
    ax_vv = axes[row, 0]
    im_vv = ax_vv.imshow(vv_f, cmap="gray", vmin=S1_DB_MIN, vmax=S1_DB_MAX)
    ax_vv.set_title(f"Tile {idx}  |  VV  (backscatter σ⁰)", color="white", fontsize=9)
    ax_vv.axis("off")
    cb_vv = add_colorbar(fig, ax_vv, im_vv, "σ⁰  [dB]",
                         ticks=[-30,-25,-20,-15,-10,-5,0,5],
                         ticklabels=["-30","-25","-20\n(water thresh)","-15","-10","-5","0","+5"])
    vv_v = vv_f[np.isfinite(vv_f)]
    ax_vv.text(0.02,0.97,
               f"μ = {vv_v.mean():.1f} dB\nσ = {vv_v.std():.1f} dB\np5 = {np.percentile(vv_v,5):.1f} dB",
               transform=ax_vv.transAxes, color="white", fontsize=6.5, va="top",
               bbox=dict(fc="#00000099",ec="none",pad=3))

    # Panel B: VH viridis with dB colorbar
    ax_vh = axes[row, 1]
    im_vh = ax_vh.imshow(vh_f, cmap="viridis", vmin=S1_DB_MIN, vmax=S1_DB_MAX)
    ax_vh.set_title(f"Tile {idx}  |  VH  (backscatter σ⁰)", color="white", fontsize=9)
    ax_vh.axis("off")
    cb_vh = add_colorbar(fig, ax_vh, im_vh, "σ⁰  [dB]",
                         ticks=[-30,-25,-20,-15,-10,-5,0,5],
                         ticklabels=["-30","-25","-20\n(KI gate)","-15","-10","-5","0","+5"])
    vh_v = vh_f[np.isfinite(vh_f)]
    ax_vh.text(0.02,0.97,
               f"μ = {vh_v.mean():.1f} dB\nσ = {vh_v.std():.1f} dB\np5 = {np.percentile(vh_v,5):.1f} dB",
               transform=ax_vh.transAxes, color="white", fontsize=6.5, va="top",
               bbox=dict(fc="#00000099",ec="none",pad=3))

    # Panel C: VV/VH composite with legend
    ax_fc = axes[row, 2]
    ax_fc.imshow(make_rgb_sar(vv_f, vh_f))
    ax_fc.set_title(f"Tile {idx}  |  Composite  VV(R) / VH(G)", color="white", fontsize=9)
    ax_fc.axis("off")
    lp = [
        mpatches.Patch(color=(1.0, 0.9, 0.0), label="Land (high VV & VH → yellow)"),
        mpatches.Patch(color=(0.0, 0.0, 0.0), label="Water (low VV & VH → black)"),
        mpatches.Patch(color=(0.7, 0.2, 0.0), label="Urban (VV > VH → red-orange)"),
        mpatches.Patch(color=(0.0, 0.5, 0.0), label="Vegetation (VH > VV → green)"),
    ]
    ax_fc.legend(handles=lp, loc="lower right", fontsize=6,
                 facecolor="#111", edgecolor="#555", labelcolor="white")
    ax_fc.text(0.02,0.97,f"{DATE_STR}\n{AREA_NAME}",
               transform=ax_fc.transAxes, color="white", fontsize=6, va="top", alpha=0.8,
               bbox=dict(fc="#00000099",ec="none",pad=2))

plt.tight_layout()
out = MASK_DIR.parent / "vis_sar_all_tiles.png"
plt.savefig(str(out), dpi=130, bbox_inches="tight", facecolor=BG)
plt.show()
print(f"Saved to {out.name}")

---
## 5 · Optical Visualization — True-Color & False-Color (all tiles)
Nodata-aware percentile stretch with gamma correction ensures dark tiles render correctly.


In [ ]:
N_OPT = len(opt_tiles)
fig, axes = plt.subplots(N_OPT, 2, figsize=(11, 5.5*N_OPT), facecolor=BG)
if N_OPT == 1: axes = axes[np.newaxis, :]

fig.suptitle(
    f"  Optical Tiles — True-Color  ·  False-Color NIR\n{AREA_NAME}   ·   {DATE_STR}",
    color="white", fontsize=13, fontweight="bold", y=1.002
)

for row, tif in enumerate(opt_tiles):
    idx   = int(tif.stem.split("_")[1])
    bands = load_opt_bands(tif)
    b2    = bands.get("B2", np.zeros((2,2)))
    vfrac = np.mean(b2>0)*100

    for ax in axes[row]: ax.set_facecolor(BG)

    # True-color
    ax_tc = axes[row, 0]
    rgb_tc = make_rgb_opt(bands, bands.get("__mask__"))
    ax_tc.imshow(rgb_tc)
    ax_tc.set_title(f"Tile {idx}  |  True-Color  B4/B3/B2", color="white", fontsize=9)
    ax_tc.axis("off")
    ax_tc.text(0.02,0.97,
               f"Valid pixels: {vfrac:.1f}%\n{DATE_STR}",
               transform=ax_tc.transAxes, color="white", fontsize=6.5, va="top",
               bbox=dict(fc="#00000099",ec="none",pad=3))
    lp_tc = [
        mpatches.Patch(color=(0.1,0.25,0.65), label="Water (dark blue)"),
        mpatches.Patch(color=(0.3,0.60,0.25), label="Vegetation (green)"),
        mpatches.Patch(color=(0.75,0.70,0.60),label="Urban / Bare soil (tan)"),
        mpatches.Patch(color=(1.00,1.00,1.00), label="Cloud (white)"),
        mpatches.Patch(color=(0.05,0.05,0.10), label="No-data (black)"),
    ]
    ax_tc.legend(handles=lp_tc, loc="lower right", fontsize=6,
                 facecolor="#111", edgecolor="#555", labelcolor="white")

    # False-color NIR
    ax_fc = axes[row, 1]
    rgb_fc = make_fc_opt(bands, bands.get("__mask__"))
    ax_fc.imshow(rgb_fc)
    ax_fc.set_title(f"Tile {idx}  |  False-Color  B12/B8/B4  (SWIR₂·NIR·Red)", color="white", fontsize=9)
    ax_fc.axis("off")
    ax_fc.text(0.02,0.97,
               f"Valid pixels: {vfrac:.1f}%\n{DATE_STR}",
               transform=ax_fc.transAxes, color="white", fontsize=6.5, va="top",
               bbox=dict(fc="#00000099",ec="none",pad=3))
    lp_fc = [
        mpatches.Patch(color=(0.80,0.15,0.15), label="Vegetation (NIR very bright → red)"),
        mpatches.Patch(color=(0.05,0.10,0.55), label="Water (NIR low → dark blue)"),
        mpatches.Patch(color=(0.60,0.50,0.40), label="Urban / Bare (tan/brown)"),
        mpatches.Patch(color=(1.00,1.00,1.00), label="Cloud (white)"),
    ]
    ax_fc.legend(handles=lp_fc, loc="lower right", fontsize=6,
                 facecolor="#111", edgecolor="#555", labelcolor="white")

plt.tight_layout()
out = MASK_DIR.parent / "vis_optical_all_tiles.png"
plt.savefig(str(out), dpi=130, bbox_inches="tight", facecolor=BG)
plt.show()
print(f"Saved to {out.name}")

---
## 6 · Batch Annotation (all paired tiles)


In [ ]:
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

results = {}
for tile_idx in tqdm(paired, desc="Annotating"):
    sp = SAR_DIR / f"tile_{tile_idx}.tif"
    op = OPT_DIR / f"tile_{tile_idx}.tif"
    vv_r, vh_r, profile = load_sar(sp)
    bands = load_opt_bands(op)
    sl,sc_prob,sm,vv_f,vh_f,sdwi = annotate_sar(vv_r, vh_r)
    ol,oc_prob,cl,sh,clr,cfrac,om,ix = annotate_optical(bands)
    fl,wp,s2w = fuse_labels(sl,sc_prob,ol,oc_prob,cfrac)
    fpct = round(100*np.sum(fl==1)/max(np.sum(fl>=0),1),2)
    prof2 = profile.copy()
    prof2.update(dtype=rasterio.int16, count=1, nodata=-1)
    with rasterio.open(MASK_DIR/f"tile_{tile_idx}.tif","w",**prof2) as dst:
        dst.write(fl.astype(np.int16), 1)
    results[tile_idx] = dict(
        sar_label=sl, sar_conf=sc_prob, sar_meta=sm,
        opt_label=ol, opt_conf=oc_prob, opt_meta=om,
        fused_label=fl, water_prob=wp,
        cloud=cl, shadow=sh, clear=clr,
        cloud_frac=cfrac, s2_weight=s2w,
        fused_water_pct=fpct,
        vv_f=vv_f, vh_f=vh_f, sdwi=sdwi,
        bands=bands, idx=ix,
    )
print(f"\nAnnotated {len(results)} paired tiles saved to {MASK_DIR.name}/")

---
## 7 · Per-Tile Annotation Dashboard
**9 panels** with colorbar thermometers on every index and mask.
- **Row 1**: SAR composite · Optical RGB · MNDWI · AWEInsh · SDWI
- **Row 2**: SAR Mask · Optical Mask · Fused Mask · Water Probability · Cloud/Shadow


In [ ]:
CLD_CMAP = ListedColormap(["#1a2a1a", "#5a5a5a", "#e0e0e0"])  # clear, shadow, cloud
CLD_NORM = BoundaryNorm([-0.5,0.5,1.5,2.5], CLD_CMAP.N)

for tile_idx, res in results.items():
    BG2  = "#0d0d1a"
    fig  = plt.figure(figsize=(26, 12), facecolor=BG2)
    gs   = gridspec.GridSpec(2, 5, figure=fig,
                             hspace=0.40, wspace=0.38,
                             left=0.02, right=0.98,
                             top=0.88,  bottom=0.04)

    bands = res["bands"]
    cl, sh = res["cloud"], res["shadow"]
    ix     = res["idx"]

    # row 0

    # 0,0  SAR composite
    ax = fig.add_subplot(gs[0,0])
    ax.imshow(make_rgb_sar(res["vv_f"], res["vh_f"]))
    ax.set_title("SAR  VV(R)/VH(G) Composite", color="white", fontsize=8.5)
    ax.axis("off")
    ax.text(0.02,0.97,
            f"KI thresh: {res['sar_meta']['ki_thresh']:.1f} dB\n"
            f"Eff thresh: {res['sar_meta']['eff_thresh']:.1f} dB",
            transform=ax.transAxes, color="cyan", fontsize=6.5, va="top",
            bbox=dict(fc="#00000099",ec="none",pad=2))
    lp_sar = [
        mpatches.Patch(color=(1.0,0.9,0.0), label="Land"),
        mpatches.Patch(color=(0.0,0.0,0.0), label="Water"),
        mpatches.Patch(color=(0.7,0.2,0.0), label="Urban"),
    ]
    ax.legend(handles=lp_sar, loc="lower right", fontsize=5.5,
              facecolor="#111", edgecolor="#444", labelcolor="white")

    # 0,1  Optical RGB
    ax1 = fig.add_subplot(gs[0,1])
    ax1.imshow(make_rgb_opt(bands, bands.get("__mask__")))
    ax1.set_title("Optical True-Color  B4/B3/B2", color="white", fontsize=8.5)
    ax1.axis("off")
    ax1.text(0.02,0.97,
             f"Cloud: {res['cloud_frac']*100:.1f}%",
             transform=ax1.transAxes, color="yellow", fontsize=6.5, va="top",
             bbox=dict(fc="#00000099",ec="none",pad=2))
    lp_opt = [
        mpatches.Patch(color=(0.1,0.25,0.65), label="Water"),
        mpatches.Patch(color=(0.3,0.60,0.25), label="Vegetation"),
        mpatches.Patch(color=(0.75,0.70,0.60),label="Urban/Bare"),
        mpatches.Patch(color=(1.00,1.00,1.00), label="Cloud"),
    ]
    ax1.legend(handles=lp_opt, loc="lower right", fontsize=5.5,
               facecolor="#111", edgecolor="#444", labelcolor="white")

    # 0,2  MNDWI
    ax2 = fig.add_subplot(gs[0,2])
    mndwi = ix["mndwi"]
    im2   = ax2.imshow(mndwi, cmap="RdYlBu", vmin=-0.6, vmax=0.6)
    ax2.set_title("MNDWI  (B3 − B11) / (B3 + B11)", color="white", fontsize=8.5)
    ax2.axis("off")
    add_colorbar(fig, ax2, im2,
                 "MNDWI (B3-B11)\n> 0.2 → water",
                 ticks=[-0.6,-0.3,0.0,0.2,0.4,0.6],
                 ticklabels=["-0.6\nDry land","-0.3","0","0.2\nThresh","0.4","0.6\nOpen water"])
    ax2.text(0.02,0.97,f"mean={res['opt_meta']['mndwi_mean']:.3f}",
             transform=ax2.transAxes, color="white", fontsize=6.5, va="top",
             bbox=dict(fc="#00000099",ec="none",pad=2))

    # 0,3  AWEInsh
    ax3 = fig.add_subplot(gs[0,3])
    ani  = ix["aweinsh"]
    im3  = ax3.imshow(ani, cmap="coolwarm", vmin=-0.5, vmax=0.5)
    ax3.set_title("AWEInsh  (no-shadow AWEI)", color="white", fontsize=8.5)
    ax3.axis("off")
    add_colorbar(fig, ax3, im3,
                 "AWEInsh\n> 0 → water",
                 ticks=[-0.5,-0.25,0.0,0.25,0.5],
                 ticklabels=["-0.5\nDry","-0.25","0\nThresh","0.25","0.5\nWater"])
    ax3.text(0.02,0.97,f"mean={res['opt_meta']['awei_nsh_mean']:.3f}",
             transform=ax3.transAxes, color="white", fontsize=6.5, va="top",
             bbox=dict(fc="#00000099",ec="none",pad=2))

    # 0,4  SDWI
    ax4 = fig.add_subplot(gs[0,4])
    sdwi  = res["sdwi"]
    vlo   = float(np.nanpercentile(sdwi,2))
    vhi   = float(np.nanpercentile(sdwi,98))
    im4   = ax4.imshow(sdwi, cmap="plasma", vmin=vlo, vmax=vhi)
    ax4.set_title("SDWI  (SAR dual-pol index)", color="white", fontsize=8.5)
    ax4.axis("off")
    t5 = np.linspace(vlo, vhi, 5)
    tl5 = [f"{t:.1f}" if t != min(t5, key=lambda x: abs(x-SDWI_THRESH)) else f"{t:.1f}\nThresh" for t in t5]
    add_colorbar(fig, ax4, im4,
                 f"SDWI\n> {SDWI_THRESH} → water",
                 ticks=list(t5), ticklabels=tl5)
    ax4.text(0.02,0.97,f"mean={res['sar_meta']['sdwi_mean']:.1f}",
             transform=ax4.transAxes, color="white", fontsize=6.5, va="top",
             bbox=dict(fc="#00000099",ec="none",pad=2))

    # row 1

    # 1,0  SAR Water Mask
    ax5 = fig.add_subplot(gs[1,0])
    ax5.imshow(res["sar_label"], cmap=WATER_CMAP, norm=WATER_NORM, interpolation="nearest")
    ax5.set_title(f"SAR Water Mask   ({res['sar_meta']['sar_water_pct']}% water)",
                  color="white", fontsize=8.5)
    ax5.axis("off")
    add_water_colorbar(fig, ax5, "SAR label")
    ax5.text(0.02,0.97,
             f"KI: {res['sar_meta']['ki_thresh']:.1f} dB\n"
             f"Lee ENL: {LEE_ENL}  W: {LEE_WINDOW}px",
             transform=ax5.transAxes, color="white", fontsize=6, va="top",
             bbox=dict(fc="#00000099",ec="none",pad=2))

    # 1,1  Optical Water Mask
    ax6 = fig.add_subplot(gs[1,1])
    opt_vis = res["opt_label"].copy().astype(np.float32)
    opt_vis[cl|sh] = -1  # cloud/shadow shown as nodata
    ax6.imshow(opt_vis, cmap=WATER_CMAP, norm=WATER_NORM, interpolation="nearest")
    ax6.set_title(f"Optical Water Mask   ({res['opt_meta']['opt_water_pct']}% water)",
                  color="white", fontsize=8.5)
    ax6.axis("off")
    add_water_colorbar(fig, ax6, "OPT label")
    ax6.text(0.02,0.97,
             f"Majority vote ≥2/3 indices\nNDVI veto > {NDVI_VEG_THRESH}",
             transform=ax6.transAxes, color="white", fontsize=6, va="top",
             bbox=dict(fc="#00000099",ec="none",pad=2))

    # 1,2  Fused Water Mask
    ax7 = fig.add_subplot(gs[1,2])
    ax7.imshow(res["fused_label"], cmap=WATER_CMAP, norm=WATER_NORM, interpolation="nearest")
    ax7.set_title(f"Fused Water Mask   ({res['fused_water_pct']}% water)",
                  color="white", fontsize=8.5)
    ax7.axis("off")
    add_water_colorbar(fig, ax7, "Fused")
    ax7.text(0.02,0.97,
             f"S2 weight = (1 − {res['cloud_frac']:.2f})²\n= {res['s2_weight']:.3f}\n"
             f"SAR weight = 1.0 (always)",
             transform=ax7.transAxes, color="white", fontsize=6, va="top",
             bbox=dict(fc="#00000099",ec="none",pad=2))

    # 1,3  Water Probability (Jet)
    ax8 = fig.add_subplot(gs[1,3])
    im8 = ax8.imshow(res["water_prob"], cmap="jet", vmin=0, vmax=1)
    ax8.set_title("Fusion Water Probability", color="white", fontsize=8.5)
    ax8.axis("off")
    add_colorbar(fig, ax8, im8,
                 "P(water)\n0 = land  ·  1 = water",
                 ticks=[0.0, 0.25, 0.5, 0.75, 1.0],
                 ticklabels=["0.0\n(Land)", "0.25", "0.5\n(Thresh)", "0.75", "1.0\n(Water)"])

    # 1,4  Cloud + Shadow Mask
    ax9 = fig.add_subplot(gs[1,4])
    cld_vis = np.zeros(cl.shape, dtype=np.int8)
    cld_vis[sh] = 1
    cld_vis[cl] = 2
    ax9.imshow(cld_vis, cmap=CLD_CMAP, norm=CLD_NORM, interpolation="nearest")
    ax9.set_title(f"Cloud + Shadow Mask   ({res['cloud_frac']*100:.1f}% obscured)",
                  color="white", fontsize=8.5)
    ax9.axis("off")
    div9 = make_axes_locatable(ax9)
    cax9 = div9.append_axes("right", size="5%", pad=0.04)
    cb9  = fig.colorbar(plt.cm.ScalarMappable(norm=CLD_NORM, cmap=CLD_CMAP), cax=cax9)
    cb9.set_ticks([0, 1, 2])
    cb9.set_ticklabels(["Clear", "Shadow", "Cloud"])
    cb9.set_label("HOT + Blue/SWIR\n+ NIR-drop test", color="white", fontsize=6)
    cb9.ax.yaxis.set_tick_params(labelsize=6, labelcolor="white", color="white")
    cb9.outline.set_edgecolor("#555")
    ax9.text(0.02,0.97,
             f"HOT thresh: {HOT_THRESH}\nBlue/SWIR: {BLUE_SWIR_THRESH}\nNIR drop: {NIR_DROP_THRESH}",
             transform=ax9.transAxes, color="white", fontsize=6, va="top",
             bbox=dict(fc="#00000099",ec="none",pad=2))

    # figure title
    fig.suptitle(
        f"  Tile {tile_idx}  ·  {AREA_NAME}  ·  {DATE_STR}\n"
        " Row 1: SAR imagery  ·  Optical RGB  ·  MNDWI  ·  AWEInsh  ·  SDWI  "
        "      Row 2: SAR Mask  ·  Optical Mask  ·  Fused Mask  ·  Water Prob  ·  Cloud Mask ",
        color="white", fontsize=9.5, fontweight="bold"
    )

    out_p = MASK_DIR.parent / f"vis_tile_{tile_idx}_dashboard.png"
    plt.savefig(str(out_p), dpi=140, bbox_inches="tight", facecolor=BG2)
    plt.show()
    print(f"Saved to {out_p.name}")

---
## 8 · Summary Statistics


In [ ]:
rows = []
for k,v in results.items():
    rows.append({"Tile":k,
                 "SAR Water %":   v["sar_meta"]["sar_water_pct"],
                 "OPT Water %":   v["opt_meta"]["opt_water_pct"],
                 "Fused Water %": v["fused_water_pct"],
                 "Cloud %":       round(v["cloud_frac"]*100,1),
                 "S2 Weight":     round(v["s2_weight"],3),
                 "KI Thresh dB":  v["sar_meta"]["ki_thresh"],
                 "SDWI mean":     v["sar_meta"]["sdwi_mean"],
                 "MNDWI mean":    v["opt_meta"]["mndwi_mean"]})
df = pd.DataFrame(rows).set_index("Tile").sort_index()
print(df.to_string())
print("Describe ")
print(df.describe().round(2))

fig, axes = plt.subplots(1,3, figsize=(16,3.5), facecolor=BG)
for ax,(col,clr) in zip(axes,[("Fused Water %","#1e90ff"),("Cloud %","#d0d0d0"),("S2 Weight","#ffa500")]):
    ax.bar(df.index.astype(str), df[col], color=clr, edgecolor="#333")
    ax.set_title(col, color="white", fontsize=9)
    ax.set_facecolor(BG)
    ax.tick_params(colors="white", labelsize=7)
    ax.set_xlabel("Tile ID", color="white", fontsize=7)
    for sp in ax.spines.values(): sp.set_edgecolor("#444")
fig.suptitle(f"Dataset Summary  ·  {AREA_NAME}  ·  {DATE_STR}",
             color="white", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.savefig(str(MASK_DIR.parent/"vis_summary_stats.png"),
            dpi=140, bbox_inches="tight", facecolor=BG)
plt.show()